---
## 11. Advanced Analyses

### 11.1 Hub Edge Analysis

In [ ]:
# Define hubs as edges with both high degree AND high betweenness
degree_threshold = stats_df['degree'].quantile(0.90)
betweenness_threshold = stats_df['betweenness'].quantile(0.90)

hub_edges = stats_df[
    (stats_df['degree'] > degree_threshold) & 
    (stats_df['betweenness'] > betweenness_threshold)
]

print("="*60)
print("HUB EDGE ANALYSIS")
print("="*60)
print(f"Hub definition: top 10% in BOTH degree AND betweenness")
print(f"Number of hub edges: {len(hub_edges)}")
print(f"\nHub edges:")
print(hub_edges[['edge_label', 'degree', 'betweenness', 'community']].head(15))

# Visualize hubs
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(stats_df['degree'], stats_df['betweenness'], 
           alpha=0.5, s=20, label='All edges')
ax.scatter(hub_edges['degree'], hub_edges['betweenness'], 
           alpha=0.8, s=100, c='red', marker='*', label='Hub edges')
ax.axvline(degree_threshold, color='red', linestyle='--', alpha=0.5)
ax.axhline(betweenness_threshold, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('Degree', fontsize=12)
ax.set_ylabel('Betweenness Centrality', fontsize=12)
ax.set_title('Hub Edge Identification', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 11.2 Weight vs Centrality Correlation

In [ ]:
# Test correlation between original edge weight and edge network centrality
corr_deg, p_deg = spearmanr(stats_df['original_weight'], stats_df['degree'])
corr_bet, p_bet = spearmanr(stats_df['original_weight'], stats_df['betweenness'])

print("="*60)
print("WEIGHT VS CENTRALITY CORRELATION")
print("="*60)
print(f"Original weight vs degree:      r={corr_deg:+.3f}, p={p_deg:.3e}")
print(f"Original weight vs betweenness: r={corr_bet:+.3f}, p={p_bet:.3e}")
print("\nInterpretation:")
if abs(corr_deg) < 0.3:
    print("  • Weak correlation suggests edge topology != original weight")
elif abs(corr_deg) < 0.7:
    print("  • Moderate correlation suggests some relationship")
else:
    print("  • Strong correlation suggests edge topology reflects original weight")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(stats_df['original_weight'], stats_df['degree'], alpha=0.5, s=20)
ax.set_xlabel('Original Edge Weight', fontsize=11)
ax.set_ylabel('Edge Degree', fontsize=11)
ax.set_title(f'Weight vs Degree\n(r={corr_deg:.3f}, p={p_deg:.3e})', 
             fontsize=12, fontweight='bold')
ax.grid(alpha=0.3)

ax = axes[1]
ax.scatter(stats_df['original_weight'], stats_df['betweenness'], alpha=0.5, s=20)
ax.set_xlabel('Original Edge Weight', fontsize=11)
ax.set_ylabel('Edge Betweenness', fontsize=11)
ax.set_title(f'Weight vs Betweenness\n(r={corr_bet:.3f}, p={p_bet:.3e})', 
             fontsize=12, fontweight='bold')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 11.3 Community Composition Analysis

In [ ]:
# Analyze which nodes contribute most to each community
print("="*60)
print("COMMUNITY COMPOSITION ANALYSIS")
print("="*60)

for comm_idx in range(len(communities)):
    # Get edges in this community
    comm_edges = [i for i, c in enumerate(communities.membership) if c == comm_idx]
    
    if len(comm_edges) < 3:
        continue
    
    # Count node participation
    node_counts = {}
    for edge_idx in comm_edges:
        source, target = edge_nodes[edge_idx]
        node_counts[source] = node_counts.get(source, 0) + 1
        node_counts[target] = node_counts.get(target, 0) + 1
    
    # Top contributing nodes
    top_nodes = sorted(node_counts.items(), key=lambda x: x[1], reverse=True)[:5]
    
    print(f"\nCommunity {comm_idx} ({len(comm_edges)} edges):")
    print(f"  Top participating nodes:")
    for node_idx, count in top_nodes:
        print(f"    {node_labels[node_idx]:40s} ({count} edges)")

---
## 12. Export Results

Save all results for further analysis or sharing.